In [41]:
import os
import json
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from openai import OpenAI
import ollama

In [42]:
load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [43]:
DATA_PATH = "../data"

documents = []

for file in os.listdir(DATA_PATH):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(DATA_PATH, file))
        documents.extend(loader.load())

print("Total pages loaded:", len(documents))

Total pages loaded: 9


In [44]:
for i, doc in enumerate(documents):
    print(i, len(doc.page_content))

0 0
1 0
2 0
3 0
4 0
5 0
6 0
7 0
8 0


In [45]:
print(documents[0])

page_content='' metadata={'source': '../data\\doc1.pdf', 'page': 0}


In [46]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "]
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 0


In [47]:
from langchain_community.document_loaders import PyPDFLoader

documents = []

for file in os.listdir(DATA_PATH):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(DATA_PATH, file))
        docs = loader.load()
        
        # DEBUG
        print(file, "pages:", len(docs))
        
        documents.extend(docs)

doc1.pdf pages: 2
doc2.pdf pages: 4
doc3.pdf pages: 3


In [48]:
print(documents[0])

page_content='' metadata={'source': '../data\\doc1.pdf', 'page': 0}


In [49]:
from langchain_community.document_loaders import PyMuPDFLoader

documents = []

for file in os.listdir(DATA_PATH):
    if file.endswith(".pdf"):
        loader = PyMuPDFLoader(os.path.join(DATA_PATH, file))
        documents.extend(loader.load())

print("Total pages loaded:", len(documents))

Total pages loaded: 9


In [50]:
print(documents[0])

page_content='' metadata={'source': '../data\\doc1.pdf', 'file_path': '../data\\doc1.pdf', 'page': 0, 'total_pages': 2, 'format': 'PDF 1.7', 'title': 'doc1.md - Google Docs', 'author': 'Subrata Das', 'subject': '', 'keywords': '', 'creator': '', 'producer': 'Microsoft: Print To PDF', 'creationDate': "D:20260323230807+05'30'", 'modDate': "D:20260323230807+05'30'", 'trapped': ''}


In [51]:
print(documents[0].page_content[:300])

In [52]:
import os
from pdf2image import convert_from_path
import pytesseract
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Tesseract path
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# Poppler path
POPPLER_PATH = r"C:\Users\subra\Downloads\poppler-25.12.0\Library\bin"
DATA_PATH = "../data"

documents = []

# 🔹 STEP 1: PDF → Images → OCR → Documents
for file in os.listdir(DATA_PATH):
    if file.endswith(".pdf"):
        pdf_path = os.path.join(DATA_PATH, file)

        print(f"Processing: {file}")

        # Convert PDF to images
        images = convert_from_path(
            pdf_path,
            poppler_path=POPPLER_PATH  # remove if PATH is set
        )

        for i, image in enumerate(images):
            text = pytesseract.image_to_string(image)

            import re

            text = re.sub(r"Q:.*?A:", "", text)  # remove Q&A patterns
            # Clean text (important)
            text = " ".join(text.split())

            if text:  # avoid empty pages
                documents.append(
                    Document(
                        page_content=text,
                        metadata={
                            "source": file,
                            "page": i
                        }
                    )
                )

print("Total documents:", len(documents))

Processing: doc1.pdf
Processing: doc2.pdf
Processing: doc3.pdf
Total documents: 9


In [53]:
print("\nSample extracted text:\n")
print(documents[0].page_content[:500]) #OCR working :) image->text conversion successful!


Sample extracted text:

INDECIMAL — Company Overview & Customer Journey (Internal Reference) Version: 1.0 Audience: Support, Sales, Product, and Al Assistant Knowledge Base Last Updated: 2025-12-21 1) One-line Summary Indecimal provides end-to-end home construction support with transparent pricing, quality assurance, and structured project tracking from inquiry to handover. 2) What Indecimal Promises (Customer-Facing Commitments) Indecimal positions its offering as building “confidence” through commitment, not just con


In [54]:
import re

def clean_text(text):
    # 1. Normalize whitespace
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    
    # 2. Fix common OCR mistakes
    replacements = {
        "saft": "sqft",
        "saqft": "sqft",
        "sq ft": "sqft",
        "rs.": "Rs",
        "₹": "Rs",
        "=": "",
        "%": "",
        "—": "-",
        "–": "-",
    }
    
    for wrong, correct in replacements.items():
        text = text.replace(wrong, correct)
    
    # 3. Fix numbers like "1,995" → keep as is (IMPORTANT)
    # Do NOT remove commas inside numbers
    
    # 4. Remove strange characters BUT keep useful ones
    text = re.sub(r"[^\w\s\.,:/\-\(\)]", "", text)
    
    # 5. Fix spacing issues
    text = re.sub(r"\s+([.,])", r"\1", text)
    
    return text.strip()

In [55]:
cleaned_documents = []
for doc in documents:
    cleaned_text = clean_text(doc.page_content)
    if cleaned_text:  # avoid empty text
        cleaned_documents.append(
            Document(
                page_content=cleaned_text,
                metadata=doc.metadata
            )
        )

In [56]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "]
)

chunks = text_splitter.split_documents(cleaned_documents)

print("\nTotal chunks:", len(chunks))


Total chunks: 33


In [57]:
print("Sample chunk:\n")
print(chunks[0].page_content)

Sample chunk:

INDECIMAL - Company Overview  Customer Journey (Internal Reference) Version: 1.0 Audience: Support, Sales, Product, and Al Assistant Knowledge Base Last Updated: 2025-12-21 1) One-line Summary Indecimal provides end-to-end home construction support with transparent pricing, quality assurance, and structured project tracking from inquiry to handover


In [58]:
from langchain.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\subra\anaconda3\envs\rag_env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [59]:
vectorstore = FAISS.from_documents(chunks, embedding)

In [60]:
vectorstore.save_local("../faiss_index")
print("FAISS index saved")

FAISS index saved


In [61]:
vectorstore = FAISS.load_local(
    "../faiss_index",
    embedding,
    allow_dangerous_deserialization=True
)

In [62]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [63]:
from langchain_community.llms import Ollama
llm = Ollama(model="phi3")

In [64]:
def run_rag(query):
    
    docs = retriever.get_relevant_documents(query)
    
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
    You are a strict QA assistant.

    Answer ONLY the given question using the context below.

    Rules:
    - Do NOT include other questions or answers from the context
    - Do NOT repeat multiple Q&A pairs
    - Give a short, direct answer
    - If answer is not found, say: "Not found in context"

    Context:
    {context}

    Question:
    {query}

    Answer:
"""

    response = llm.invoke(prompt)
    print(context)
    return {
        "question": query,
        "answer": response,
        "sources": [doc.metadata for doc in docs]
    }


In [65]:
result = run_rag("What is indecimal pricing?")

print("\nAnswer:\n")
print(result["answer"])

print("\nSources:\n")
for s in result["sources"]:
    print(s)

. Q: How does Indecimal reduce hidden surprises in pricing A: The flow emphasizes receiving plans that include detailed design and cost plans with transparent pricing and no hidden surprises. Q: What makes contractor payments stage-based A: The Why Us section states payments are released only after verified completion. (End of document)

. 2) What Indecimal Promises (Customer-Facing Commitments) Indecimal positions its offering as building confidence through commitment, not just contracts. The approach emphasizes clarity and trust across the construction and ownership journey. 3) What We Strive For (Operating Principles) 1. Smooth Construction Experience - Step-by-step support throughout the project. Best and Competitive Pricing - Fair pricing with no hidden charges

INDECIMAL - Customer Protection Policies, Quality System, and Guarantees (Internal Reference) Version: 1.0 Audience: Support, Ops, Project Management, Al Assistant Knowledge Base Last Updated: 2025-12-21 1) Payment Safety 

In [66]:
docs = retriever.get_relevant_documents("What is indecimal pricing?")

for i, doc in enumerate(docs):
    print(f"\n--- Chunk {i+1} ---\n")
    print(doc.page_content)


--- Chunk 1 ---

. Q: How does Indecimal reduce hidden surprises in pricing A: The flow emphasizes receiving plans that include detailed design and cost plans with transparent pricing and no hidden surprises. Q: What makes contractor payments stage-based A: The Why Us section states payments are released only after verified completion. (End of document)

--- Chunk 2 ---

. 2) What Indecimal Promises (Customer-Facing Commitments) Indecimal positions its offering as building confidence through commitment, not just contracts. The approach emphasizes clarity and trust across the construction and ownership journey. 3) What We Strive For (Operating Principles) 1. Smooth Construction Experience - Step-by-step support throughout the project. Best and Competitive Pricing - Fair pricing with no hidden charges

--- Chunk 3 ---

INDECIMAL - Customer Protection Policies, Quality System, and Guarantees (Internal Reference) Version: 1.0 Audience: Support, Ops, Project Management, Al Assistant Knowle

In [67]:
#improving retrieval by reranker
from sentence_transformers import CrossEncoder
reranker_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

c:\Users\subra\anaconda3\envs\rag_env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [68]:
def rerank_docs(docs, query):
    
    # Create query-doc pairs
    pairs = [[query, doc.page_content] for doc in docs]
    
    # Get relevance scores
    scores = reranker_model.predict(pairs)
    
    # Sort by score (descending)
    ranked_docs = [doc for _, doc in sorted(
        zip(scores, docs),
        key=lambda x: x[0],
        reverse=True
    )]
    
    return ranked_docs

In [69]:
def run_rag(query):
    
    # 🔹 Step 1: Retrieve (k=5)
    docs = retriever.get_relevant_documents(query)
    
    # 🔹 Step 2: Rerank (🔥 cross-encoder)
    docs = rerank_docs(docs, query)
    
    # 🔹 Step 3: Take top 3
    docs = docs[:3]
    
    # 🔹 Step 4: Build context
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # 🔹 Step 5: Prompt
    prompt = f"""
    Answer ONLY the given question using the context below.

    Rules:
    - Do not include unrelated information
    - If not found, say "Not found in context"

    Context:
    {context}

    Question:
    {query}

    Answer:
    """
    
    response = llm.invoke(prompt)
    
    return {
        "answer": response,
        "sources": [doc.metadata for doc in docs]
    }

In [70]:
def debug_pipeline(query):
    
    print("\n================ QUERY ================\n")
    print(query)
    
    # 🔹 Step 1: Retrieve (FAISS)
    retrieved_docs = retriever.get_relevant_documents(query)
    
    print("\n=========== RETRIEVED (FAISS) ===========\n")
    for i, doc in enumerate(retrieved_docs):
        print(f"\n--- Retrieved {i+1} ---")
        print(doc.page_content[:200])
        print("Source:", doc.metadata)
    
    # 🔹 Step 2: Rerank
    reranked_docs = rerank_docs(retrieved_docs, query)
    
    print("\n=========== RERANKED (CROSS-ENCODER) ===========\n")
    for i, doc in enumerate(reranked_docs):
        print(f"\n--- Reranked {i+1} ---")
        print(doc.page_content[:200])
        print("Source:", doc.metadata)
    
    return reranked_docs

In [71]:
debug_pipeline("What is indecimal pricing?")    


================ QUERY ================

What is indecimal pricing?

=========== RETRIEVED (FAISS) ===========


--- Retrieved 1 ---
. Q: How does Indecimal reduce hidden surprises in pricing A: The flow emphasizes receiving plans that include detailed design and cost plans with transparent pricing and no hidden surprises. Q: What 
Source: {'source': 'doc1.pdf', 'page': 1}

--- Retrieved 2 ---
. 2) What Indecimal Promises (Customer-Facing Commitments) Indecimal positions its offering as building confidence through commitment, not just contracts. The approach emphasizes clarity and trust acr
Source: {'source': 'doc1.pdf', 'page': 0}

--- Retrieved 3 ---
INDECIMAL - Customer Protection Policies, Quality System, and Guarantees (Internal Reference) Version: 1.0 Audience: Support, Ops, Project Management, Al Assistant Knowledge Base Last Updated: 2025-12
Source: {'source': 'doc3.pdf', 'page': 0}

--- Retrieved 4 ---
INDECIMAL - Company Overview  Customer Journey (Internal Reference) Versio

[Document(metadata={'source': 'doc1.pdf', 'page': 0}, page_content='INDECIMAL - Company Overview  Customer Journey (Internal Reference) Version: 1.0 Audience: Support, Sales, Product, and Al Assistant Knowledge Base Last Updated: 2025-12-21 1) One-line Summary Indecimal provides end-to-end home construction support with transparent pricing, quality assurance, and structured project tracking from inquiry to handover'),
 Document(metadata={'source': 'doc1.pdf', 'page': 1}, page_content='. Q: How does Indecimal reduce hidden surprises in pricing A: The flow emphasizes receiving plans that include detailed design and cost plans with transparent pricing and no hidden surprises. Q: What makes contractor payments stage-based A: The Why Us section states payments are released only after verified completion. (End of document)'),
 Document(metadata={'source': 'doc1.pdf', 'page': 0}, page_content='. 2) What Indecimal Promises (Customer-Facing Commitments) Indecimal positions its offering as build

In [72]:
result = run_rag("What is indecimal pricing?")

print("\nAnswer:\n")
print(result["answer"])

print("\nSources:\n")
for s in result["sources"]:
    print(s)


Answer:

Indecimal provides transparent pricing in their home construction services, emphasizing clear communication of costs from the start to avoid any unexpected expenses.

Sources:

{'source': 'doc1.pdf', 'page': 0}
{'source': 'doc1.pdf', 'page': 1}
{'source': 'doc1.pdf', 'page': 0}


In [73]:
result = run_rag("What are the package rates?")

print("\nAnswer:\n")
print(result["answer"])

print("\nSources:\n")
for s in result["sources"]:
    print(s)


Answer:

- Essential Package Rate: 1,851 /sqft including GST.

- Premier (Most Popular) Package Rate: 1,995 /sqft including GST.

- Infinia Package Rate: 2,250 /sqft including GST.

- Pinnacle Package Rate: 2,450 /sqft including GST.

Sources:

{'source': 'doc2.pdf', 'page': 0}
{'source': 'doc2.pdf', 'page': 0}
{'source': 'doc2.pdf', 'page': 1}


In [74]:
debug_pipeline("What are the package rates?")    


================ QUERY ================

What are the package rates?

=========== RETRIEVED (FAISS) ===========


--- Retrieved 1 ---
INDECIMAL - Package Comparison  Specification Wallets (Internal Reference) Version: 1.0 Audience: Sales, Estimation, Al Assistant Knowledge Base Last Updated: 2025-12-21 1) Package Pricing (Indicative
Source: {'source': 'doc2.pdf', 'page': 0}

--- Retrieved 2 ---
. Block Work - Solid concrete blocks 6 (external)  4 (internal) - Pricing guidance shown as: - 6: up to 40 (/- 3) per block
Source: {'source': 'doc2.pdf', 'page': 0}

--- Retrieved 3 ---
2,000 - Premier: Parryware/Hindware or equivalent up to 2,500 - Infinia: Jaquar/Essco or equivalent up to 3,500 - Pinnacle: Jaquar/Essco or equivalent up to 4,000 4) Bathroom (Wallet Allowances) Note:
Source: {'source': 'doc2.pdf', 'page': 1}

--- Retrieved 4 ---
Bharthi, or equivalent up to 370/bag - Infinia: Birla Super, Ramco, or equivalent up to 390/bag - Pinnacle: ACC, Ultratech, Ramco, or equivalent up to

[Document(metadata={'source': 'doc2.pdf', 'page': 0}, page_content='INDECIMAL - Package Comparison  Specification Wallets (Internal Reference) Version: 1.0 Audience: Sales, Estimation, Al Assistant Knowledge Base Last Updated: 2025-12-21 1) Package Pricing (Indicative / Per Sqft) These are shown as per-sqft package rates (inclusive of GST) on the public comparison page: - Essential: 1,851 /sqft (incl. GST) - Premier (Most Popular): 1,995 /sqft (incl. GST) - - Infinia: 2,250 /sqft (incl. GST) - Pinnacle: 2,450 /sqft (incl'),
 Document(metadata={'source': 'doc2.pdf', 'page': 0}, page_content='Bharthi, or equivalent up to 370/bag - Infinia: Birla Super, Ramco, or equivalent up to 390/bag - Pinnacle: ACC, Ultratech, Ramco, or equivalent up to 400/bag Aggregates - 20mm  40mm aggregates across all packages'),
 Document(metadata={'source': 'doc2.pdf', 'page': 1}, page_content='- 4: up to 33 (/- 3) per block RCC Mix - M20 or M25 (RCC design mix as advised by the structural engineer) Ceiling He